In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters \
chromadb pypdf sentence-transformers transformers accelerate

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

from transformers import pipeline

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
loader = PyPDFLoader("rpa_blueprism.pdf")
documents = loader.load()

print("Pages:", len(documents))

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
db = Chroma.from_documents(chunks, embedding)

retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
llm_pipeline = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    max_length=512
)

In [ ]:
def rag_query(query):
    docs = retriever.get_relevant_documents(query)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    Answer the question based only on the context.

    Context:
    {context}

    Question:
    {query}
    """

    result = llm_pipeline(prompt)[0]["generated_text"]
    return result

In [ ]:
def summarize():
    docs = retriever.get_relevant_documents("Give a full summary of the document")
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    Create:
    - Bullet point summary
    - Key concepts
    - Important definitions

    From:
    {context}
    """

    result = llm_pipeline(prompt)[0]["generated_text"]
    return result

In [ ]:
# This section re-defines dependencies to make this cell runnable independently.
# In a typical Colab workflow, you would ensure all preceding cells defining
# `loader`, `documents`, `splitter`, `chunks`, `embedding`, `db`, `retriever`,
# `llm_pipeline`, `rag_query`, and `summarize` are executed first.

# Re-import necessary modules if not globally available
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# Re-define components for retriever and its dependencies
loader = PyPDFLoader("rpa_blueprism.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(chunks, embedding)
retriever = db.as_retriever(search_kwargs={"k": 3})

# Re-define llm_pipeline (assuming it was the last modified component)
llm_pipeline = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base",
    max_length=512
)

# Re-define rag_query function
def rag_query(query):
    docs = retriever.get_relevant_documents(query)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    Answer the question based only on the context.

    Context:
    {context}

    Question:
    {query}
    """

    result = llm_pipeline(prompt)[0]["generated_text"]
    return result

# Re-define summarize function
def summarize():
    docs = retriever.get_relevant_documents("Give a full summary of the document")
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    Create:
    - Bullet point summary
    - Key concepts
    - Important definitions

    From:
    {context}
    """

    result = llm_pipeline(prompt)[0]["generated_text"]
    return result


print(rag_query("What is RPA?"))
print("\n-----------------\n")
print(rag_query("Explain Blue Prism"))
print("\n-----------------\n")
print(summarize())